In [1]:
import pandas as pd
import numpy as np
import json
import os

In [2]:
#path
folder = r"C:\Users\LLR User\NHS_Project"

# files from 2014-2024
files = [
    ('2014-15', 'hosp-epis-stat-admi-diag-2014-15-tab.xlsx'),
    ('2015-16', 'hosp-epis-stat-admi-diag-2015-16-tab.xlsx'),
    ('2016-17', 'hosp-epis-stat-admi-diag-2016-17-tab.xlsx'),
    ('2017-18', 'hosp-epis-stat-admi-diag-2017-18-tab.xlsx'),
    ('2018-19', 'hosp-epis-stat-admi-diag-2018-19-tab.xlsx'),
    ('2019-20', 'hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx'),
    ('2020-21', 'hosp-epis-stat-admi-diag-2020-21-tab.xlsx'),
    ('2021-22', 'hosp-epis-stat-admi-diag-2021-22-tab.xlsx'),
    ('2022-23', 'hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx'),
    ('2023-24', 'hosp-epis-stat-admi-diag-2023-24-tab.xlsx'),
]

print("Files found:")
for year, fname in files:
    full_path = os.path.join(folder, fname)
    exists = os.path.exists(full_path)
    print(f"  {year}: {'✓' if exists else '✗ NOT FOUND'} {fname}")

Files found:
  2014-15: ✓ hosp-epis-stat-admi-diag-2014-15-tab.xlsx
  2015-16: ✓ hosp-epis-stat-admi-diag-2015-16-tab.xlsx
  2016-17: ✓ hosp-epis-stat-admi-diag-2016-17-tab.xlsx
  2017-18: ✓ hosp-epis-stat-admi-diag-2017-18-tab.xlsx
  2018-19: ✓ hosp-epis-stat-admi-diag-2018-19-tab.xlsx
  2019-20: ✓ hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx
  2020-21: ✓ hosp-epis-stat-admi-diag-2020-21-tab.xlsx
  2021-22: ✓ hosp-epis-stat-admi-diag-2021-22-tab.xlsx
  2022-23: ✓ hosp-epis-stat-admi-diag-2022-23-tab_V2.xlsx
  2023-24: ✓ hosp-epis-stat-admi-diag-2023-24-tab.xlsx


In [3]:
def extract_data(filepath, year_label):
    wb_sheets = pd.ExcelFile(filepath).sheet_names
    sheet_name = 'Primary Diagnosis Summary' if 'Primary Diagnosis Summary' in wb_sheets else 'Primary diagnosis - summary'
    
    from openpyxl import load_workbook
    wb = load_workbook(filepath, read_only=True)
    ws = wb[sheet_name]
    
    rows = []
    header_row_idx = None
    
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if any('Admission' in str(v) for v in row if v):
            header_row_idx = i
            break
    
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i <= header_row_idx:
            continue
        if len([v for v in row if v not in (None,'')]) < 5:
            continue
        code = row[0]
        desc = row[1]
        if code in (None,'') or str(code).strip() == 'Total':
            continue
        code_str = str(code).strip().replace('‡ ','')
        if not code_str or code_str[0].isdigit():
            continue
        def safe(v):
            try: return float(v) if v not in (None,'') else None
            except: return None
        rows.append({
            'year': year_label, 'code': code_str,
            'description': str(desc).strip() if desc else '',
            'admissions': safe(row[3]), 'emergency': safe(row[7]),
            'waiting_list': safe(row[8]), 'mean_wait': safe(row[11]),
            'mean_los': safe(row[13]), 'mean_age': safe(row[15]),
        })
    return rows

# Extract all years
all_data = []
for year, fname in files:
    rows = extract_data(os.path.join(folder, fname), year)
    all_data.extend(rows)
    print(f"{year}: {len(rows)} diagnosis categories extracted")

df = pd.DataFrame(all_data)
for col in ['admissions','emergency','waiting_list','mean_wait','mean_los','mean_age']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"\nTotal rows: {len(df)}")
print(f"Years: {sorted(df['year'].unique())}")

2014-15: 149 diagnosis categories extracted
2015-16: 149 diagnosis categories extracted
2016-17: 148 diagnosis categories extracted
2017-18: 148 diagnosis categories extracted
2018-19: 148 diagnosis categories extracted
2019-20: 215 diagnosis categories extracted
2020-21: 215 diagnosis categories extracted
2021-22: 215 diagnosis categories extracted
2022-23: 215 diagnosis categories extracted
2023-24: 216 diagnosis categories extracted

Total rows: 1818
Years: ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']


In [4]:
def get_chapter(code):
    if not isinstance(code, str): return 'Other'
    c = code[0].upper()
    return {
        'A':'Infectious & Parasitic','B':'Infectious & Parasitic',
        'C':'Neoplasms','D':'Neoplasms',
        'E':'Endocrine & Metabolic','F':'Mental & Behavioural',
        'G':'Nervous System','H':'Eye / Ear','I':'Circulatory',
        'J':'Respiratory','K':'Digestive','L':'Skin',
        'M':'Musculoskeletal','N':'Genitourinary',
        'O':'Pregnancy & Childbirth','R':'Symptoms & Signs',
        'S':'Injury & Poisoning','T':'Injury & Poisoning',
        'Z':'Health Status & Contact',
    }.get(c, 'Other')

df['chapter'] = df['code'].apply(get_chapter)

# Aggregate by chapter and year
ch_yr = df.groupby(['chapter','year']).agg(
    admissions=('admissions','sum'),
    emergency=('emergency','sum'),
    waiting_list=('waiting_list','sum'),
    mean_wait=('mean_wait','mean'),
    mean_los=('mean_los','mean'),
).reset_index()

# Base year = 2023-24, comparison = 2014-15
ch_2023 = ch_yr[ch_yr['year']=='2023-24'].set_index('chapter')
ch_2014 = ch_yr[ch_yr['year']=='2014-15'].set_index('chapter')
common_ch = ch_2023.index.intersection(ch_2014.index)

ch = ch_2023.loc[common_ch].copy()
ch['waitlist_growth'] = ((ch['waiting_list'] - ch_2014.loc[common_ch,'waiting_list']) / ch_2014.loc[common_ch,'waiting_list'] * 100).round(1).clip(-100,800)
ch['emergency_rate']  = (ch['emergency'] / ch['admissions'] * 100).round(1)
ch['log_admissions']  = np.log10(ch['admissions'].clip(1)).round(2)
ch['mean_wait']       = ch['mean_wait'].clip(0,200).round(1)
ch['mean_los']        = ch['mean_los'].clip(0,40).round(1)
ch = ch.dropna(subset=['waiting_list','mean_wait','mean_los','emergency_rate'])
ch = ch[ch['waiting_list'] > 0]
ch = ch[~ch.index.isin(['Other'])]

print(f"Chapters ready: {len(ch)}")
print("\nTop chapters by waiting list size:")
print(ch[['waiting_list','mean_wait','mean_los','emergency_rate','waitlist_growth']].sort_values('waiting_list', ascending=False).head(8).to_string())

Chapters ready: 16

Top chapters by waiting list size:
                         waiting_list  mean_wait  mean_los  emergency_rate  waitlist_growth
chapter                                                                                    
Digestive                   1437642.0       83.1       5.2            26.4             16.7
Neoplasms                   1420324.0       47.1       5.7             8.8             49.0
Eye / Ear                    832294.0      107.4       1.8             6.3             24.3
Musculoskeletal              711895.0      140.8       6.8            26.5            -21.5
Symptoms & Signs             455036.0       51.0       2.8            68.5              8.2
Genitourinary                414051.0       98.3       3.4            45.9             -9.9
Circulatory                  274790.0       66.6       7.3            64.4            -24.6
Health Status & Contact      228467.0       71.4       5.0             6.4            -11.4


In [5]:
# Get 2023-24 individual codes as subcategories
df_2023 = df[df['year']=='2023-24'].copy().reset_index(drop=True)
df_2014_idx = df[df['year']=='2014-15'].set_index('code')

df_2023 = df_2023.dropna(subset=['admissions','waiting_list','mean_wait','mean_los'])
df_2023 = df_2023[df_2023['waiting_list'] > 100]
df_2023['waitlist_2014']   = df_2023['code'].map(df_2014_idx['waiting_list'])
df_2023['waitlist_growth'] = ((df_2023['waiting_list'] - df_2023['waitlist_2014']) / df_2023['waitlist_2014'] * 100).round(1).clip(-100,800)
df_2023['emergency_rate']  = (df_2023['emergency'] / df_2023['admissions'] * 100).round(1)
df_2023['log_admissions']  = np.log10(df_2023['admissions'].clip(1)).round(2)
df_2023['mean_wait']       = df_2023['mean_wait'].clip(0,200).round(1)
df_2023['mean_los']        = df_2023['mean_los'].clip(0,40).round(1)
df_2023                    = df_2023[df_2023['chapter'].isin(ch.index)]

# Top 3 per chapter by waiting list size
sub_records = []
for chap in ch.index:
    subset = df_2023[df_2023['chapter'] == chap].nlargest(3, 'waiting_list')
    for _, row in subset.iterrows():
        desc = str(row['description'])
        sub_records.append({
            'type': 'sub', 'code': row['code'],
            'description': desc[:50], 'full_description': desc,
            'chapter': chap,
            'log_admissions': round(float(row['log_admissions']), 2),
            'admissions': int(row['admissions']),
            'waiting_list': int(row['waiting_list']),
            'mean_wait': round(float(row['mean_wait']), 1),
            'mean_los': round(float(row['mean_los']), 1),
            'emergency_rate': round(float(row['emergency_rate']), 1),
            'waitlist_growth': round(float(row['waitlist_growth']), 1) if not pd.isna(row['waitlist_growth']) else 0,
        })

print(f"Subcategories ready: {len(sub_records)}")
print("\nSample subcategories:")
for r in sub_records[:6]:
    print(f"  {r['code']} [{r['chapter']}]: wl={r['waiting_list']:,}, wait={r['mean_wait']}d")

Subcategories ready: 48

Sample subcategories:
  I30-I52 [Circulatory]: wl=101,455, wait=92.8d
  I20-I25 [Circulatory]: wl=72,659, wait=68.6d
  I80-I89 [Circulatory]: wl=45,708, wait=114.4d
  K55-K64 [Digestive]: wl=421,362, wait=57.6d
  K20-K31 [Digestive]: wl=322,959, wait=43.7d
  K50-K52 [Digestive]: wl=189,128, wait=76.1d


In [10]:
# Build chapter records
chapter_records = []
for chapter, row in ch.iterrows():
    chapter_records.append({
        'type': 'chapter', 'code': chapter, 'description': chapter,
        'full_description': chapter, 'chapter': chapter,
        'log_admissions': round(float(row['log_admissions']), 2),
        'admissions': int(row['admissions']),
        'waiting_list': int(row['waiting_list']),
        'mean_wait': round(float(row['mean_wait']), 1),
        'mean_los': round(float(row['mean_los']), 1),
        'emergency_rate': round(float(row['emergency_rate']), 1),
        'waitlist_growth': round(float(row['waitlist_growth']), 1),
    })

# Save JSON data
with open(os.path.join(folder, 'nhs_data.json'), 'w') as f:
    json.dump({'chapters': chapter_records, 'subcategories': sub_records}, f)

print(f"Chapter records: {len(chapter_records)}")
print(f"Subcategory records: {len(sub_records)}")
print(f"\nData saved to: {folder}\\nhs_data.json")

# Now read the JSON back and build the HTML
with open(os.path.join(folder, 'nhs_data.json')) as f:
    data = json.load(f)

html = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>NHS Waiting List Pressure — Parallel Coordinates</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: "Segoe UI", sans-serif; background: #0b0f1a; color: #e2e8f0; }
header { padding: 14px 26px 10px; background: linear-gradient(90deg,#0b0f1a,#111827); border-bottom: 1px solid #1e2d3d; }
header h1 { font-size: 16px; font-weight: 700; color: #f1f5f9; }
header .sub { font-size: 11.5px; color: #64748b; margin-top:3px; font-style:italic; }
header .meta { font-size: 10px; color: #475569; margin-top:2px; }
#controls { display:flex; flex-wrap:wrap; gap:7px; align-items:center; padding:7px 26px; background:#0f172a; border-bottom:1px solid #1e2d3d; }
#controls label { font-size:11px; color:#64748b; }
.btn { padding:3px 11px; border-radius:20px; border:1px solid #1e2d3d; background:#1e293b; color:#94a3b8; font-size:11px; cursor:pointer; transition:all .15s; }
.btn.active { background:#431407; border-color:#f97316; color:#fdba74; }
.btn:hover { border-color:#475569; }
#lvl-toggle { display:flex; gap:5px; margin-left:auto; }
.lvl-btn { padding:3px 11px; border-radius:20px; border:1px solid #1e2d3d; background:#1e293b; color:#94a3b8; font-size:11px; cursor:pointer; }
.lvl-btn.active { background:#14532d; border-color:#22c55e; color:#86efac; }
#count { font-size:11px; color:#475569; }
#main { display:flex; height:calc(100vh - 112px); }
#chart { flex:1; position:relative; overflow:hidden; }
svg { width:100%; height:100%; }
.grid { stroke:#1e293b; stroke-width:1; stroke-dasharray:3,6; }
.axis { stroke:#334155; stroke-width:1.5; }
.tick { stroke:#334155; stroke-width:1; }
.alabel { font-size:11.5px; fill:#cbd5e1; font-weight:700; text-anchor:middle; }
.aunit { font-size:9px; fill:#475569; text-anchor:middle; }
.tlabel { font-size:8.5px; fill:#475569; text-anchor:middle; }
.cline { fill:none; stroke-width:2.8; stroke-opacity:.75; cursor:pointer; transition:stroke-opacity .1s,stroke-width .1s; }
.cline:hover,.cline.pin { stroke-opacity:1; stroke-width:4; filter:drop-shadow(0 0 4px currentColor); }
.cline.dim { stroke-opacity:.05; pointer-events:none; }
.sline { fill:none; stroke-width:1.2; stroke-opacity:.5; stroke-dasharray:5,3; cursor:pointer; transition:stroke-opacity .1s,stroke-width .1s; }
.sline:hover,.sline.pin { stroke-opacity:.95; stroke-width:2; }
.sline.dim { stroke-opacity:.04; pointer-events:none; }
#tt { position:absolute; pointer-events:none; display:none; background:#0f172a; border:1px solid #1e2d3d; border-radius:10px; padding:12px 16px; font-size:12px; max-width:295px; z-index:99; box-shadow:0 12px 40px rgba(0,0,0,.7); }
.tt-title { font-weight:700; color:#f1f5f9; font-size:13px; margin-bottom:3px; line-height:1.35; }
.tt-code { font-size:10px; margin-bottom:8px; }
.tt-row { display:flex; justify-content:space-between; gap:16px; color:#64748b; margin:3px 0; font-size:11.5px; }
.tt-val { color:#e2e8f0; font-weight:600; }
.tt-type { display:inline-block; padding:1px 7px; border-radius:8px; font-size:9.5px; margin-top:6px; }
#sidebar { width:185px; padding:12px 10px; background:#0f172a; border-left:1px solid #1e2d3d; overflow-y:auto; flex-shrink:0; }
#sidebar h3 { font-size:10px; color:#475569; text-transform:uppercase; letter-spacing:1px; margin-bottom:8px; }
.li { display:flex; align-items:center; gap:7px; padding:3px 5px; border-radius:5px; cursor:pointer; margin-bottom:2px; transition:background .1s; }
.li:hover { background:#1e293b; }
.li.off { opacity:.25; }
.dot { width:9px; height:9px; border-radius:50%; flex-shrink:0; }
.lname { font-size:10px; color:#cbd5e1; flex:1; line-height:1.3; }
#infobar { position:absolute; bottom:8px; left:50%; transform:translateX(-50%); background:#0f172acc; border:1px solid #1e293b; border-radius:6px; padding:4px 12px; font-size:10px; color:#475569; pointer-events:none; white-space:nowrap; }
</style>
</head>
<body>
<header>
  <h1>NHS Hospital Admissions Waiting List Pressure by Diagnosis Chapter</h1>
  <div class="sub">Which admission categories place the greatest pressure on NHS waiting lists, and has this grown over 10 years?</div>
  <div class="meta">Primary Diagnosis Summary · England · 2014–2024 · 16 ICD Chapters + top 3 subcategories · Source: NHS Digital / ONS HES</div>
</header>
<div id="controls">
  <label>Filter:</label>
  <button class="btn active" data-f="all">All Chapters</button>
  <button class="btn" data-f="high-wait">Long Wait (&gt;100d)</button>
  <button class="btn" data-f="high-emerg">High Emergency (&gt;70%)</button>
  <button class="btn" data-f="mental">Mental Health</button>
  <button class="btn" data-f="musculo">Musculoskeletal</button>
  <button class="btn" data-f="growing">Growing Backlog</button>
  <div id="lvl-toggle">
    <span style="font-size:11px;color:#64748b;align-self:center">Show:</span>
    <button class="lvl-btn active" id="btn-ch">Chapters</button>
    <button class="lvl-btn" id="btn-sub">+ Subcategories</button>
  </div>
  <span id="count"></span>
</div>
<div id="main">
  <div id="chart">
    <svg id="svg"></svg>
    <div id="tt"></div>
    <div id="infobar">Hover a line for details · Click to pin · Toggle subcategories top-right · Click legend to isolate chapter</div>
  </div>
  <div id="sidebar"><h3>ICD Chapter</h3><div id="sb-body"></div></div>
</div>
<script>
''' + f"const CHAPTERS = {json.dumps(data['chapters'])};\n" \
    + f"const SUBS = {json.dumps(data['subcategories'])};\n" + '''
const COLORS = {
  'Infectious & Parasitic':'#f97316','Neoplasms':'#a855f7',
  'Endocrine & Metabolic':'#eab308','Mental & Behavioural':'#ec4899',
  'Nervous System':'#8b5cf6','Eye / Ear':'#06b6d4','Circulatory':'#ef4444',
  'Respiratory':'#3b82f6','Digestive':'#22c55e','Skin':'#f59e0b',
  'Musculoskeletal':'#84cc16','Genitourinary':'#14b8a6',
  'Pregnancy & Childbirth':'#f472b6','Symptoms & Signs':'#94a3b8',
  'Injury & Poisoning':'#fbbf24','Health Status & Contact':'#64748b',
};
const AXES = [
  {key:'log_admissions', label:'Total Admissions',     unit:'(log\u2081\u2080 scale)',          fmt:v=>Math.round(Math.pow(10,v)).toLocaleString()},
  {key:'waiting_list',   label:'Waiting List Size',    unit:'(patients queued, 2024)', fmt:v=>Math.round(v/1000)+'k'},
  {key:'mean_wait',      label:'Mean Wait Time',       unit:'(days)',                  fmt:v=>v.toFixed(0)+'d'},
  {key:'mean_los',       label:'Mean Length of Stay',  unit:'(days, once admitted)',   fmt:v=>v.toFixed(1)+'d'},
  {key:'emergency_rate', label:'Emergency Rate',       unit:'(% bypassing waitlist)',  fmt:v=>v.toFixed(0)+'%'},
  {key:'waitlist_growth',label:'Waitlist Growth',      unit:'(% change 2014\u21922024)',fmt:v=>(v>0?'+':'')+v.toFixed(0)+'%'},
];
const M={top:68,bottom:26,left:55,right:18};
let activeChapters=new Set(CHAPTERS.map(d=>d.chapter));
let activeFilter='all', pinned=null, showSubs=false;
const svgEl=document.getElementById('svg');
const ttEl=document.getElementById('tt');
const caEl=document.getElementById('chart');
function sz(){const r=caEl.getBoundingClientRect();return{W:r.width,H:r.height};}
function scl(W,H){
  const iW=W-M.left-M.right,iH=H-M.top-M.bottom;
  const xPos=AXES.map((_,i)=>M.left+i/(AXES.length-1)*iW);
  const ys={};
  const all=[...CHAPTERS,...SUBS];
  AXES.forEach(ax=>{
    const vs=all.map(d=>d[ax.key]).filter(v=>v!=null&&!isNaN(v));
    const mn=Math.min(...vs),mx=Math.max(...vs),pad=(mx-mn)*0.07;
    ys[ax.key]={min:mn-pad,max:mx+pad};
  });
  return{xPos,ys,iH};
}
function yp(key,val,ys,iH){
  const s=ys[key];
  if(val==null||isNaN(val))return M.top+iH/2;
  return M.top+iH-((val-s.min)/(s.max-s.min))*iH;
}
function mkPath(d,xPos,ys,iH){
  return 'M'+AXES.map((ax,i)=>`${xPos[i]},${yp(ax.key,d[ax.key],ys,iH)}`).join('L');
}
function passes(d){
  if(!activeChapters.has(d.chapter)) return false;
  if(activeFilter==='high-wait'  && d.mean_wait<=100)     return false;
  if(activeFilter==='high-emerg' && d.emergency_rate<=70)  return false;
  if(activeFilter==='mental'     && d.chapter!=='Mental & Behavioural') return false;
  if(activeFilter==='musculo'    && d.chapter!=='Musculoskeletal')       return false;
  if(activeFilter==='growing'    && d.waitlist_growth<=20) return false;
  return true;
}
function render(){
  const{W,H}=sz();
  svgEl.setAttribute('viewBox',`0 0 ${W} ${H}`);
  const{xPos,ys,iH}=scl(W,H);
  let html='';
  for(let t=0;t<=4;t++){
    const y=M.top+t/4*iH;
    html+=`<line class="grid" x1="${xPos[0]}" y1="${y}" x2="${xPos[xPos.length-1]}" y2="${y}"/>`;
  }
  AXES.forEach((ax,i)=>{
    const x=xPos[i],yt=M.top,yb=M.top+iH;
    html+=`<line class="axis" x1="${x}" y1="${yt}" x2="${x}" y2="${yb}"/>`;
    html+=`<text class="alabel" x="${x}" y="${yt-36}">${ax.label}</text>`;
    html+=`<text class="aunit"  x="${x}" y="${yt-22}">${ax.unit}</text>`;
    const s=ys[ax.key];
    for(let t=0;t<=4;t++){
      const val=s.min+t/4*(s.max-s.min),y=M.top+iH-t/4*iH;
      html+=`<line class="tick" x1="${x-4}" y1="${y}" x2="${x+4}" y2="${y}"/>`;
      html+=`<text class="tlabel" x="${x}" y="${y+11}">${ax.fmt(val)}</text>`;
    }
  });
  if(showSubs){
    SUBS.filter(d=>!passes(d)).forEach(d=>{
      html+=`<path class="sline dim" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#475569'}" data-code="${d.code}" data-t="sub"/>`;
    });
    SUBS.filter(passes).forEach(d=>{
      const cls=d.code===pinned?'sline pin':'sline';
      html+=`<path class="${cls}" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#475569'}" data-code="${d.code}" data-t="sub"/>`;
    });
  }
  CHAPTERS.filter(d=>!passes(d)).forEach(d=>{
    html+=`<path class="cline dim" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#475569'}" data-code="${d.code}" data-t="ch"/>`;
  });
  CHAPTERS.filter(passes).forEach(d=>{
    const cls=d.code===pinned?'cline pin':'cline';
    html+=`<path class="${cls}" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#475569'}" data-code="${d.code}" data-t="ch"/>`;
  });
  svgEl.innerHTML=html;
  document.getElementById('count').textContent=`${CHAPTERS.filter(passes).length}/${CHAPTERS.length} chapters`;
  svgEl.querySelectorAll('.cline:not(.dim),.sline:not(.dim)').forEach(el=>{
    el.addEventListener('mouseenter',onHover);
    el.addEventListener('mouseleave',onLeave);
    el.addEventListener('click',onClick);
  });
}
function tooltip(d){
  const gc=d.waitlist_growth>50?'#f87171':d.waitlist_growth<0?'#4ade80':'#fbbf24';
  const ec=d.emergency_rate>70?'#f87171':d.emergency_rate>40?'#fbbf24':'#4ade80';
  const badge=d.type==='chapter'
    ?`<span class="tt-type" style="background:#1e3a5f;color:#93c5fd">Chapter aggregate</span>`
    :`<span class="tt-type" style="background:#1a2e1a;color:#86efac">Subcategory &bull; ${d.chapter}</span>`;
  return `
    <div class="tt-title">${d.full_description}</div>
    <div class="tt-code" style="color:${COLORS[d.chapter]||'#94a3b8'}">${d.code}</div>
    <div class="tt-row"><span>Total Admissions (2023-24)</span><span class="tt-val">${d.admissions.toLocaleString()}</span></div>
    <div class="tt-row"><span>Waiting List Size</span><span class="tt-val">${d.waiting_list.toLocaleString()}</span></div>
    <div class="tt-row"><span>Mean Wait Time</span><span class="tt-val">${d.mean_wait} days</span></div>
    <div class="tt-row"><span>Mean Length of Stay</span><span class="tt-val">${d.mean_los} days</span></div>
    <div class="tt-row"><span>Emergency Rate</span><span class="tt-val" style="color:${ec}">${d.emergency_rate}%</span></div>
    <div class="tt-row"><span>Waitlist Growth 2014→2024</span><span class="tt-val" style="color:${gc}">${d.waitlist_growth>0?'+':''}${d.waitlist_growth}%</span></div>
    ${badge}`;
}
function findData(code,t){return t==='ch'?CHAPTERS.find(d=>d.code===code):SUBS.find(d=>d.code===code);}
function onHover(e){
  const d=findData(e.target.dataset.code,e.target.dataset.t); if(!d) return;
  if(d.code!==pinned){e.target.style.strokeOpacity='1';e.target.style.strokeWidth=d.type==='chapter'?'4':'2';}
  ttEl.innerHTML=tooltip(d); ttEl.style.display='block'; moveTT(e);
}
function onLeave(e){
  if(e.target.dataset.code!==pinned){e.target.style.strokeOpacity='';e.target.style.strokeWidth='';}
  ttEl.style.display='none';
}
function onClick(e){pinned=pinned===e.target.dataset.code?null:e.target.dataset.code; render();}
function moveTT(e){
  const r=caEl.getBoundingClientRect();
  let x=e.clientX-r.left+14,y=e.clientY-r.top-10;
  if(x+300>r.width) x-=315;
  if(y+260>r.height) y-=260;
  ttEl.style.left=x+'px'; ttEl.style.top=y+'px';
}
svgEl.addEventListener('mousemove',e=>{if(ttEl.style.display==='block') moveTT(e);});
function buildSidebar(){
  const body=document.getElementById('sb-body');
  CHAPTERS.forEach(ch=>{
    const div=document.createElement('div'); div.className='li'; div.dataset.ch=ch.chapter;
    div.innerHTML=`<div class="dot" style="background:${COLORS[ch.chapter]||'#475569'}"></div><div class="lname">${ch.chapter}</div>`;
    div.addEventListener('click',()=>{
      if(activeChapters.size===CHAPTERS.length){activeChapters=new Set([ch.chapter]);}
      else if(activeChapters.has(ch.chapter)&&activeChapters.size===1){activeChapters=new Set(CHAPTERS.map(d=>d.chapter));}
      else{activeChapters=new Set([ch.chapter]);}
      document.querySelectorAll('.li').forEach(l=>l.classList.toggle('off',!activeChapters.has(l.dataset.ch)));
      render();
    });
    body.appendChild(div);
  });
}
document.querySelectorAll('.btn').forEach(b=>{
  b.addEventListener('click',()=>{
    document.querySelectorAll('.btn').forEach(x=>x.classList.remove('active'));
    b.classList.add('active'); activeFilter=b.dataset.f; render();
  });
});
document.getElementById('btn-ch').addEventListener('click',()=>{
  showSubs=false;
  document.getElementById('btn-ch').classList.add('active');
  document.getElementById('btn-sub').classList.remove('active');
  render();
});
document.getElementById('btn-sub').addEventListener('click',()=>{
  showSubs=true;
  document.getElementById('btn-sub').classList.add('active');
  document.getElementById('btn-ch').classList.remove('active');
  render();
});
buildSidebar(); render();
window.addEventListener('resize',render);
</script>
</body>
</html>'''

# Save the HTML file
output_path = os.path.join(folder, 'nhs_waiting_list_viz.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"✓ Visualization saved to:\n  {output_path}")
print(f"\nOpen that file in your browser to see the chart!")

Chapter records: 16
Subcategory records: 48

Data saved to: C:\Users\LLR User\NHS_Project\nhs_data.json
✓ Visualization saved to:
  C:\Users\LLR User\NHS_Project\nhs_waiting_list_viz.html

Open that file in your browser to see the chart!


In [7]:
# # Build chapter records
# chapter_records = []
# for chapter, row in ch.iterrows():
#     chapter_records.append({
#         'type': 'chapter', 'code': chapter, 'description': chapter,
#         'full_description': chapter, 'chapter': chapter,
#         'log_admissions': round(float(row['log_admissions']), 2),
#         'admissions': int(row['admissions']),
#         'waiting_list': int(row['waiting_list']),
#         'mean_wait': round(float(row['mean_wait']), 1),
#         'mean_los': round(float(row['mean_los']), 1),
#         'emergency_rate': round(float(row['emergency_rate']), 1),
#         'waitlist_growth': round(float(row['waitlist_growth']), 1),
#     })

# # Save JSON data
# with open(os.path.join(folder, 'nhs_data.json'), 'w') as f:
#     json.dump({'chapters': chapter_records, 'subcategories': sub_records}, f)

# print(f"Chapter records: {len(chapter_records)}")
# print(f"Subcategory records: {len(sub_records)}")
# print(f"\nData saved to: {folder}\\nhs_data.json")

# # Now read the JSON back and build the HTML
# with open(os.path.join(folder, 'nhs_data.json')) as f:
#     data = json.load(f)

In [12]:
COLORS = {
    'Circulatory':            '#2B5BA8',
    'Respiratory':            '#4A90D9',
    'Neoplasms':              '#8B2FC9',
    'Mental & Behavioural':   '#C0392B',
    'Musculoskeletal':        '#27AE60',
    'Digestive':              '#E67E22',
    'Nervous System':         '#16A085',
    'Eye / Ear':              '#2980B9',
    'Endocrine & Metabolic':  '#8E44AD',
    'Genitourinary':          '#1ABC9C',
    'Skin':                   '#F39C12',
    'Infectious & Parasitic': '#D35400',
    'Injury & Poisoning':     '#7F8C8D',
    'Pregnancy & Childbirth': '#E91E8C',
    'Symptoms & Signs':       '#95A5A6',
    'Health Status & Contact':'#BDC3C7',
}

html = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>NHS Waiting List Pressure — Parallel Coordinates</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: Georgia, "Times New Roman", serif; background: #f7f9fc; color: #1a2332; }
header { padding: 18px 30px 12px; background: #ffffff; border-bottom: 2px solid #2B5BA8; }
header h1 { font-size: 20px; font-weight: 700; color: #1a2332; font-family: Georgia, serif; }
header .sub { font-size: 14px; color: #4a6080; margin-top: 4px; font-style: italic; }
header .meta { font-size: 12px; color: #7a8fa8; margin-top: 3px; font-family: "Segoe UI", Arial, sans-serif; }
#controls { display:flex; flex-wrap:wrap; gap:8px; align-items:center; padding:9px 30px; background:#eef2f8; border-bottom:1px solid #d0daea; font-family:"Segoe UI",Arial,sans-serif; }
#controls label { font-size:13px; color:#4a6080; font-weight:600; }
.btn { padding:5px 15px; border-radius:4px; border:1px solid #b0c0d8; background:#ffffff; color:#2B5BA8; font-size:13px; cursor:pointer; transition:all .15s; font-family:"Segoe UI",Arial,sans-serif; }
.btn.active { background:#2B5BA8; color:#ffffff; border-color:#2B5BA8; }
.btn:hover { background:#dce8f8; }
#lvl-toggle { display:flex; gap:6px; margin-left:auto; align-items:center; }
#lvl-label { font-size:13px; color:#4a6080; font-weight:600; }
.lvl-btn { padding:5px 15px; border-radius:4px; border:1px solid #b0c0d8; background:#ffffff; color:#27AE60; font-size:13px; cursor:pointer; font-family:"Segoe UI",Arial,sans-serif; transition:all .15s; }
.lvl-btn.active { background:#27AE60; color:#ffffff; border-color:#27AE60; }
#count { font-size:13px; color:#7a8fa8; font-family:"Segoe UI",Arial,sans-serif; }
#main { display:flex; height:calc(100vh - 125px); }
#chart { flex:1; position:relative; overflow:hidden; background:#f7f9fc; }
svg { width:100%; height:100%; }
.grid { stroke:#dde5f0; stroke-width:1; stroke-dasharray:4,6; }
.axis { stroke:#2B5BA8; stroke-width:1.5; }
.tick { stroke:#8aa0c0; stroke-width:1; }
.alabel { font-size:14px; fill:#1a2332; font-weight:700; text-anchor:middle; font-family:Georgia,serif; }
.aunit  { font-size:11px; fill:#7a8fa8; text-anchor:middle; font-family:"Segoe UI",Arial,sans-serif; }
.tlabel { font-size:10.5px; fill:#8aa0c0; text-anchor:middle; font-family:"Segoe UI",Arial,sans-serif; }
.cline { fill:none; stroke-width:2.5; stroke-opacity:.72; cursor:pointer; transition:stroke-opacity .1s,stroke-width .1s; }
.cline:hover,.cline.pin { stroke-opacity:1; stroke-width:4; }
.cline.dim { stroke-opacity:.06; pointer-events:none; }
.sline { fill:none; stroke-width:1.1; stroke-opacity:.45; stroke-dasharray:5,3; cursor:pointer; transition:stroke-opacity .1s; }
.sline:hover,.sline.pin { stroke-opacity:.9; stroke-width:2; }
.sline.dim { stroke-opacity:.04; pointer-events:none; }
#tt { position:absolute; pointer-events:none; display:none; background:#ffffff; border:1px solid #b0c0d8; border-radius:6px; padding:13px 17px; font-size:13px; max-width:300px; z-index:99; box-shadow:0 4px 20px rgba(43,91,168,.15); font-family:"Segoe UI",Arial,sans-serif; }
.tt-title { font-weight:700; color:#1a2332; font-size:14px; margin-bottom:4px; line-height:1.35; font-family:Georgia,serif; }
.tt-code { font-size:11px; margin-bottom:9px; color:#4a6080; }
.tt-row { display:flex; justify-content:space-between; gap:16px; color:#7a8fa8; margin:4px 0; font-size:12.5px; }
.tt-val { color:#1a2332; font-weight:600; }
.tt-badge { display:inline-block; padding:2px 8px; border-radius:4px; font-size:11px; margin-top:7px; }
#sidebar { width:200px; padding:14px 12px; background:#ffffff; border-left:1px solid #d0daea; overflow-y:auto; flex-shrink:0; font-family:"Segoe UI",Arial,sans-serif; }
#sidebar h3 { font-size:12px; color:#7a8fa8; text-transform:uppercase; letter-spacing:1px; margin-bottom:10px; font-weight:600; }
.li { display:flex; align-items:center; gap:7px; padding:5px 6px; border-radius:4px; cursor:pointer; margin-bottom:3px; transition:background .1s; }
.li:hover { background:#eef2f8; }
.li.off { opacity:.25; }
.dot { width:11px; height:11px; border-radius:2px; flex-shrink:0; }
.lname { font-size:11.5px; color:#2a3a50; flex:1; line-height:1.3; }
#infobar { position:absolute; bottom:10px; left:50%; transform:translateX(-50%); background:rgba(255,255,255,.9); border:1px solid #d0daea; border-radius:4px; padding:5px 14px; font-size:12px; color:#7a8fa8; pointer-events:none; white-space:nowrap; font-family:"Segoe UI",Arial,sans-serif; }
</style>
</head>
<body>
<header>
  <h1>NHS Hospital Admissions Waiting List Pressure by Diagnosis Chapter</h1>
  <div class="sub">Which admission categories place the greatest pressure on NHS waiting lists, and has this grown over 10 years?</div>
  <div class="meta">Primary Diagnosis Summary &nbsp;·&nbsp; England &nbsp;·&nbsp; 2014–2024 &nbsp;·&nbsp; 16 ICD Chapters + top 3 subcategories per chapter &nbsp;·&nbsp; Source: NHS Digital / ONS Hospital Episode Statistics</div>
</header>
<div id="controls">
  <label>Highlight:</label>
  <button class="btn active" data-f="all">All Chapters</button>
  <button class="btn" data-f="mental">Mental Health</button>
  <button class="btn" data-f="musculo">Musculoskeletal</button>
  <button class="btn" data-f="growing">Growing Backlog</button>
  <div id="lvl-toggle">
    <span id="lvl-label">Hierarchy:</span>
    <button class="lvl-btn active" id="btn-ch">Chapters only</button>
    <button class="lvl-btn" id="btn-sub">+ Subcategories</button>
  </div>
  <span id="count"></span>
</div>
<div id="main">
  <div id="chart">
    <svg id="svg"></svg>
    <div id="tt"></div>
    <div id="infobar">Hover a line for details &nbsp;·&nbsp; Click to pin &nbsp;·&nbsp; Click legend to isolate a chapter &nbsp;·&nbsp; Toggle subcategories to explore hierarchy</div>
  </div>
  <div id="sidebar">
    <h3>ICD Chapter</h3>
    <div id="sb-body"></div>
  </div>
</div>
<script>
''' + f"const CHAPTERS = {json.dumps(chapter_records)};\n" \
    + f"const SUBS = {json.dumps(sub_records)};\n" \
    + f"const COLORS = {json.dumps(COLORS)};\n" + '''
const AXES = [
  {key:'log_admissions', label:'Total Admissions',     unit:'(log\u2081\u2080 scale)',           fmt:v=>Math.round(Math.pow(10,v)).toLocaleString()},
  {key:'waiting_list',   label:'Waiting List Size',    unit:'(patients queued, 2024)',  fmt:v=>Math.round(v/1000)+'k'},
  {key:'mean_wait',      label:'Mean Wait Time',       unit:'(days)',                   fmt:v=>v.toFixed(0)+'d'},
  {key:'mean_los',       label:'Mean Length of Stay',  unit:'(days once admitted)',     fmt:v=>v.toFixed(1)+'d'},
  {key:'emergency_rate', label:'Emergency Rate',       unit:'(% bypassing waitlist)',   fmt:v=>v.toFixed(0)+'%'},
  {key:'waitlist_growth',label:'Waitlist Growth',      unit:'(% change 2014\u21922024)',fmt:v=>(v>0?'+':'')+v.toFixed(0)+'%'},
];
const M={top:80,bottom:32,left:62,right:22};
let activeChapters=new Set(CHAPTERS.map(d=>d.chapter));
let activeFilter='all', pinned=null, showSubs=false;
const svgEl=document.getElementById('svg');
const ttEl=document.getElementById('tt');
const caEl=document.getElementById('chart');
function sz(){const r=caEl.getBoundingClientRect();return{W:r.width,H:r.height};}
function scl(W,H){
  const iW=W-M.left-M.right,iH=H-M.top-M.bottom;
  const xPos=AXES.map((_,i)=>M.left+i/(AXES.length-1)*iW);
  const ys={};
  const all=[...CHAPTERS,...SUBS];
  AXES.forEach(ax=>{
    const vs=all.map(d=>d[ax.key]).filter(v=>v!=null&&!isNaN(v));
    const mn=Math.min(...vs),mx=Math.max(...vs),pad=(mx-mn)*0.07;
    ys[ax.key]={min:mn-pad,max:mx+pad};
  });
  return{xPos,ys,iH};
}
function yp(key,val,ys,iH){
  const s=ys[key];
  if(val==null||isNaN(val)) return M.top+iH/2;
  return M.top+iH-((val-s.min)/(s.max-s.min))*iH;
}
function mkPath(d,xPos,ys,iH){
  return 'M'+AXES.map((ax,i)=>`${xPos[i]},${yp(ax.key,d[ax.key],ys,iH)}`).join('L');
}
function passes(d){
  if(!activeChapters.has(d.chapter)) return false;
  if(activeFilter==='mental'  && d.chapter!=='Mental & Behavioural') return false;
  if(activeFilter==='musculo' && d.chapter!=='Musculoskeletal')       return false;
  if(activeFilter==='growing' && d.waitlist_growth<=20)               return false;
  return true;
}
function render(){
  const{W,H}=sz();
  svgEl.setAttribute('viewBox',`0 0 ${W} ${H}`);
  const{xPos,ys,iH}=scl(W,H);
  let html='';
  for(let t=0;t<=4;t++){
    const y=M.top+t/4*iH;
    html+=`<line class="grid" x1="${xPos[0]}" y1="${y}" x2="${xPos[xPos.length-1]}" y2="${y}"/>`;
  }
  AXES.forEach((ax,i)=>{
    const x=xPos[i],yt=M.top,yb=M.top+iH;
    html+=`<line class="axis" x1="${x}" y1="${yt}" x2="${x}" y2="${yb}"/>`;
    html+=`<text class="alabel" x="${x}" y="${yt-44}">${ax.label}</text>`;
    html+=`<text class="aunit"  x="${x}" y="${yt-28}">${ax.unit}</text>`;
    const s=ys[ax.key];
    for(let t=0;t<=4;t++){
      const val=s.min+t/4*(s.max-s.min),y=M.top+iH-t/4*iH;
      html+=`<line class="tick" x1="${x-4}" y1="${y}" x2="${x+4}" y2="${y}"/>`;
      html+=`<text class="tlabel" x="${x}" y="${y+14}">${ax.fmt(val)}</text>`;
    }
  });
  if(showSubs){
    SUBS.filter(d=>!passes(d)).forEach(d=>{
      html+=`<path class="sline dim" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#95A5A6'}" data-code="${d.code}" data-t="sub"/>`;
    });
    SUBS.filter(passes).forEach(d=>{
      const cls=d.code===pinned?'sline pin':'sline';
      html+=`<path class="${cls}" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#95A5A6'}" data-code="${d.code}" data-t="sub"/>`;
    });
  }
  CHAPTERS.filter(d=>!passes(d)).forEach(d=>{
    html+=`<path class="cline dim" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#95A5A6'}" data-code="${d.code}" data-t="ch"/>`;
  });
  CHAPTERS.filter(passes).forEach(d=>{
    const cls=d.code===pinned?'cline pin':'cline';
    html+=`<path class="${cls}" d="${mkPath(d,xPos,ys,iH)}" stroke="${COLORS[d.chapter]||'#95A5A6'}" data-code="${d.code}" data-t="ch"/>`;
  });
  svgEl.innerHTML=html;
  document.getElementById('count').textContent=`${CHAPTERS.filter(passes).length} / ${CHAPTERS.length} chapters shown`;
  svgEl.querySelectorAll('.cline:not(.dim),.sline:not(.dim)').forEach(el=>{
    el.addEventListener('mouseenter',onHover);
    el.addEventListener('mouseleave',onLeave);
    el.addEventListener('click',onClick);
  });
}
function tooltip(d){
  const gc=d.waitlist_growth>50?'#C0392B':d.waitlist_growth<0?'#27AE60':'#E67E22';
  const ec=d.emergency_rate>70?'#C0392B':d.emergency_rate>40?'#E67E22':'#27AE60';
  const badge=d.type==='chapter'
    ?`<span class="tt-badge" style="background:#eef2f8;color:#2B5BA8">Chapter aggregate</span>`
    :`<span class="tt-badge" style="background:#f0faf0;color:#27AE60">Subcategory &bull; ${d.chapter}</span>`;
  return `
    <div class="tt-title">${d.full_description}</div>
    <div class="tt-code">${d.code}</div>
    <div class="tt-row"><span>Total Admissions (2023-24)</span><span class="tt-val">${d.admissions.toLocaleString()}</span></div>
    <div class="tt-row"><span>Waiting List Size</span><span class="tt-val">${d.waiting_list.toLocaleString()}</span></div>
    <div class="tt-row"><span>Mean Wait Time</span><span class="tt-val">${d.mean_wait} days</span></div>
    <div class="tt-row"><span>Mean Length of Stay</span><span class="tt-val">${d.mean_los} days</span></div>
    <div class="tt-row"><span>Emergency Rate</span><span class="tt-val" style="color:${ec}">${d.emergency_rate}%</span></div>
    <div class="tt-row"><span>Waitlist Growth 2014→2024</span><span class="tt-val" style="color:${gc}">${d.waitlist_growth>0?'+':''}${d.waitlist_growth}%</span></div>
    ${badge}`;
}
function findData(code,t){return t==='ch'?CHAPTERS.find(d=>d.code===code):SUBS.find(d=>d.code===code);}
function onHover(e){
  const d=findData(e.target.dataset.code,e.target.dataset.t); if(!d) return;
  if(d.code!==pinned){e.target.style.strokeOpacity='1';e.target.style.strokeWidth=d.type==='chapter'?'4':'2';}
  ttEl.innerHTML=tooltip(d); ttEl.style.display='block'; moveTT(e);
}
function onLeave(e){
  if(e.target.dataset.code!==pinned){e.target.style.strokeOpacity='';e.target.style.strokeWidth='';}
  ttEl.style.display='none';
}
function onClick(e){pinned=pinned===e.target.dataset.code?null:e.target.dataset.code; render();}
function moveTT(e){
  const r=caEl.getBoundingClientRect();
  let x=e.clientX-r.left+14,y=e.clientY-r.top-10;
  if(x+305>r.width) x-=320;
  if(y+270>r.height) y-=270;
  ttEl.style.left=x+'px'; ttEl.style.top=y+'px';
}
svgEl.addEventListener('mousemove',e=>{if(ttEl.style.display==='block') moveTT(e);});
function buildSidebar(){
  const body=document.getElementById('sb-body');
  CHAPTERS.forEach(ch=>{
    const div=document.createElement('div'); div.className='li'; div.dataset.ch=ch.chapter;
    div.innerHTML=`<div class="dot" style="background:${COLORS[ch.chapter]||'#95A5A6'}"></div><div class="lname">${ch.chapter}</div>`;
    div.addEventListener('click',()=>{
      if(activeChapters.size===CHAPTERS.length){activeChapters=new Set([ch.chapter]);}
      else if(activeChapters.has(ch.chapter)&&activeChapters.size===1){activeChapters=new Set(CHAPTERS.map(d=>d.chapter));}
      else{activeChapters=new Set([ch.chapter]);}
      document.querySelectorAll('.li').forEach(l=>l.classList.toggle('off',!activeChapters.has(l.dataset.ch)));
      render();
    });
    body.appendChild(div);
  });
}
document.querySelectorAll('.btn').forEach(b=>{
  b.addEventListener('click',()=>{
    document.querySelectorAll('.btn').forEach(x=>x.classList.remove('active'));
    b.classList.add('active'); activeFilter=b.dataset.f; render();
  });
});
document.getElementById('btn-ch').addEventListener('click',()=>{
  showSubs=false;
  document.getElementById('btn-ch').classList.add('active');
  document.getElementById('btn-sub').classList.remove('active');
  render();
});
document.getElementById('btn-sub').addEventListener('click',()=>{
  showSubs=true;
  document.getElementById('btn-sub').classList.add('active');
  document.getElementById('btn-ch').classList.remove('active');
  render();
});
buildSidebar(); render();
window.addEventListener('resize',render);
</script>
</body>
</html>'''

output_path = os.path.join(folder, 'nhs_waiting_list_final.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"✓ Saved to: {output_path}")
print(f"\nOpen this file in your browser to see your final visualization!")

✓ Saved to: C:\Users\LLR User\NHS_Project\nhs_waiting_list_final.html

Open this file in your browser to see your final visualization!
